# 6장 — RLHF와 KL 목줄 🦮 (핵심)

5장의 REINFORCE 훈련장에 **함정 하나**를 심습니다 — 심판(RM)이 과대평가하는 해킹 답변 H.

그리고 같은 훈련을 두 번 돌립니다:
1. **목줄 없이(β=0)** — 정책이 H 로 폭주하는 것을 목격하고
2. **KL 목줄을 달고(β=2)** — 목적 r − β·KL 이 폭주를 막는 것을 확인합니다

5부 6장의 KL 발산이 안전장치로 일하는 현장입니다.

In [ ]:
# 🤖 AI 튜터 — 공부하다 막히면 오른쪽 아래 버튼 (로컬 Ollama, 없어도 학습 지장 없음)
from tutor import chat
chat('ch6')

## 1. 무대 — 함정이 있는 훈련장

- 답변 4개. **H(해킹)** 는 심판 점수가 가장 높지만(4.5) **진짜 품질은 나쁩니다**(−1) —
  심판이 좋아하는 패턴(감사 반복·과공손)만 채운 답이라고 상상하세요.
- **참조 모델 π_ref** = SFT 직후의 분포. 좋은 답을 주로 내고, H 같은 답은 거의 안 냅니다(5%).

In [ ]:
import math
import random

ACTIONS = ['A 간결·정중', 'B 장황', 'C 무성의', 'H 해킹(감사감사...)']
RM_SCORE = [3.0, 1.0, -2.0, 4.5]   # 심판(보상 모델)의 점수 — H를 과대평가!
TRUE_SCORE = [3.0, 1.0, -2.0, -1.0]  # 진짜 사람이 느끼는 품질 — H는 쓰레기
PI_REF = [0.55, 0.30, 0.10, 0.05]  # 참조 모델(SFT)의 분포 — H는 거의 안 냄

def softmax(z):
    exps = [math.exp(v) for v in z]
    s = sum(exps)
    return [e / s for e in exps]

def train(beta, episodes=400, lr=0.05, seed=42):
    """REINFORCE + KL 목줄. 갱신에 쓰는 보상 = RM 점수 − β·(log π − log π_ref)."""
    random.seed(seed)
    z = [0.0] * len(ACTIONS)
    true_curve = []
    for _ in range(episodes):
        probs = softmax(z)
        i = random.choices(range(len(ACTIONS)), weights=probs)[0]
        # KL 목줄: 원본(π_ref)보다 이 답을 과하게 자주 내면 그만큼 벌점
        leash = beta * (math.log(probs[i]) - math.log(PI_REF[i]))
        r_used = RM_SCORE[i] - leash
        for j in range(len(z)):
            grad = (1 - probs[j]) if j == i else (-probs[j])
            z[j] += lr * r_used * grad
        true_curve.append(sum(p * t for p, t in zip(softmax(z), TRUE_SCORE)))
    return softmax(z), true_curve

print('무대 준비 완료 — 답변 4개, 심판은 H를 과대평가, 참조 모델은 H를 거의 안 냄')

## 2. 실험 — 목줄 없이 vs 목줄 달고

In [ ]:
pi_free, curve_free = train(beta=0)    # 목줄 없음
pi_leash, curve_leash = train(beta=2)  # KL 목줄

print('β=0 (목줄 없음) 최종 정책:')
for name, p in zip(ACTIONS, pi_free):
    print(f'   {name:22s} {p:5.0%}')
print(f'   → 진짜 품질 기대값: {sum(p * t for p, t in zip(pi_free, TRUE_SCORE)):+.2f}')

print('\nβ=2 (KL 목줄) 최종 정책:')
for name, p in zip(ACTIONS, pi_leash):
    print(f'   {name:22s} {p:5.0%}')
print(f'   → 진짜 품질 기대값: {sum(p * t for p, t in zip(pi_leash, TRUE_SCORE)):+.2f}')

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6.5, 3.2))
plt.plot(curve_free, color='#f43f5e', linewidth=1, label='β=0 목줄 없음 → 보상 해킹')
plt.plot(curve_leash, color='#10b981', linewidth=1, label='β=2 KL 목줄 → 건강한 조련')
plt.xlabel('에피소드')
plt.ylabel('진짜 품질의 기대값')
plt.title('같은 훈련, 목줄의 유무 — 심판 점수가 아닌 진짜 품질로 본 결과')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**빨간 곡선(목줄 없음)**: 심판 점수는 계속 올랐겠지만, 진짜 품질은 곤두박질 — H(해킹)로 폭주했습니다.
**초록 곡선(목줄)**: H 를 참조 모델보다 자주 내는 순간 벌점이 커져서, 정책은 A(진짜 좋은 답)에 정착했어요.

## 3. 검증

In [ ]:
true_free = sum(p * t for p, t in zip(pi_free, TRUE_SCORE))
true_leash = sum(p * t for p, t in zip(pi_leash, TRUE_SCORE))

# 검증 1 — 목줄 없으면 해킹 답변(H)으로 폭주
assert pi_free.index(max(pi_free)) == 3, 'β=0 이면 정책이 H(해킹)로 수렴해야 한다!'
# 검증 2 — 목줄을 달면 진짜 좋은 답(A)에 정착
assert pi_leash.index(max(pi_leash)) == 0, 'β=2 면 정책이 A(간결·정중)로 수렴해야 한다!'
# 검증 3 — 진짜 품질은 목줄 쪽이 압도적으로 높다
assert true_leash > true_free + 2, 'KL 목줄이 진짜 품질을 지켜야 한다!'
print(f'✔ 확인: 목줄 없음 → H 폭주(진짜 품질 {true_free:+.2f}) / KL 목줄 → A 정착(진짜 품질 {true_leash:+.2f})')
print('✔ 심판 점수만 보면 β=0 이 승자처럼 보이지만, 사람이 보면 정반대 — 이게 KL 목줄의 존재 이유')

## 4. 관찰 정리 + 🤖 AI 연결

- **목적 = r − β·KL(π‖π_ref)** — 점수는 앞으로, 목줄은 원본 쪽으로. 코드에서는
  `RM 점수 − β·(log π − log π_ref)` 한 줄이 그 목줄이었습니다.
- 심판(RM) 점수와 진짜 품질이 갈라질 때 목줄이 생명줄이 된다 — **5부 6장 KL 발산**이
  "조련이 원본에서 멀어지지 않게 묶는 목줄"이라던 로드맵의 문장이 이 코드입니다.
- 진짜 RLHF(PPO)와 다른 점: 행동이 답변 4개가 아니라 토큰 시퀀스이고, KL도 토큰 단위로
  누적 계산되며, 안정화 장치가 더 붙습니다 — 하지만 목적함수의 뼈대는 동일.

**다음 챕터**: 7장 — 모델 세 개를 운용하는 RLHF가 부담스럽다면? 보상 모델 없이 선호 쌍에서
바로 배우는 지름길 DPO. KL 목줄은 그 수식 안에 그대로 살아 있습니다.

> 💡 궁금한 코드는 그 셀을 실행한 다음 셀에서 `from tutor import explain` 후 `explain('ch6')`.